# Transport principal domicile–travail (INSEE)

On sélectionne les actifs occupés de 15 ans ou plus, tous lieux de travail confondus. Les codes des modalités figurent dans config.yaml.

In [7]:
# Préparer les chemins, lire les réglages et créer le dossier de sortie.
import os
from pathlib import Path
import sys

# Trouver le dossier partagé depuis le dépôt ou le dossier notebooks.
racine = Path.cwd()
while not (racine / "config.yaml").is_file() and racine.parent != racine:
    dossier_insee = racine / "insee_population_navettes"
    if (dossier_insee / "config.yaml").is_file():
        racine = dossier_insee
        break
    racine = racine.parent
if not (racine / "config.yaml").is_file():
    raise FileNotFoundError("Impossible de trouver le dossier insee_population_navettes.")
os.chdir(racine)
sys.path.insert(0, str(racine))

from src.collecte import lire_config

config = lire_config()
config["hors_ligne"] = os.environ.get("RAIL_HORS_LIGNE") == "1"
propre = Path(config["dossier_propre"])
propre.mkdir(parents=True, exist_ok=True)

## Collecte

L'instantané est archivé dans `data/raw/rail2026`. La collecte suivante relit le cache.

In [8]:
# Lire les navettes INSEE brutes ; l?aper?u sert ? voir les codes des modes.
from src.insee import get_navettes

brut = get_navettes(config)
brut.head(3)

INSEE navettes : 1,587,222 lignes brutes lues ; archive et empreinte contrôlées.


,GEO,GEO_OBJECT,TRANS,AGE,EMPSTA_ENQ,WORK_AREA,RP_MEASURE,FREQ,OBS_STATUS,TIME_PERIOD,OBS_VALUE
0,001,AAV2020,1,Y_GE15,1,_T,POP,A,A,2012,220847.09555
1,001,AAV2020,2,Y_GE15,1,_T,POP,A,A,2012,416028.62459
2,001,AAV2020,3T4,Y_GE15,1,_T,POP,A,A,2012,255875.89971


## Nettoyage

Le vélo et les deux-roues motorisés forment une famille commune. Les cinq modes sont rapportés à l'ensemble des actifs occupés, y compris les actifs sans déplacement. Le seuil de 1 000 actifs intervient ensuite dans l'assemblage.

In [9]:
# Garder les deux ann?es et r?unir les modes de d?placement comme pr?vu.
from src.insee import clean_navettes

table = clean_navettes(brut, config)
display(table.head())

  Navettes 2017 : écart maximal somme des cinq modes / total = 3e-05 ; 12 alertes.
  Navettes 2023 : écart maximal somme des cinq modes / total = 3e-05 ; 0 alertes.
Navettes propres : 34,746 communes métropolitaines ; zéro publié conservé, absence conservée.


,code_commune,actifs_2017,actifs_sans_deplacement_2017,actifs_marche_2017,actifs_deux_roues_2017,actifs_voiture_2017,actifs_transports_commun_2017,alerte_navettes_2017,statut_navettes_2017,actifs_2023,actifs_sans_deplacement_2023,actifs_marche_2023,actifs_deux_roues_2023,actifs_voiture_2023,actifs_transports_commun_2023,alerte_navettes_2023,statut_navettes_2023
0,01001,346.89523,20.29790,4.04315,5.02641,305.47913,12.04865,False,A,408.42616,21.01355,7.00671,5.94218,370.45270,4.01103,False,A
1,01002,116.34568,4.08230,0.00000,0.00000,106.13992,6.12346,False,A,132.60283,11.83069,0.00000,0.97597,112.78632,7.00985,False,A
2,01004,5869.51452,241.20854,424.28783,230.48752,4169.56426,803.96637,False,A,6859.89560,257.35396,550.62490,200.31616,4876.24948,975.35110,False,A
3,01005,807.58408,39.80522,27.16817,15.55164,696.97219,28.08686,False,A,943.78774,43.58785,20.00484,8.48684,843.42729,28.28092,False,A
4,01006,50.45455,3.02727,0.00000,0.00000,46.41818,1.00909,False,A,47.41228,2.01754,0.00000,2.01754,43.37719,0.00000,False,A


## Contrôle et export

La clé ou le couple de clés identifie chaque observation. Le fichier propre garde ces types pour les jointures.

In [10]:
# V?rifier qu?il reste une ligne par commune.
assert table["code_commune"].is_unique
table.filter(regex="^actifs_2023$|^actifs_voiture_2023$").sum()

actifs_2023            2.802871e+07
actifs_voiture_2023    1.930134e+07
dtype: float64

In [11]:
# Enregistrer les effectifs nettoy?s ; les parts seront calcul?es apr?s l?assemblage.
table.to_parquet(propre / "source_navettes.parquet", index=False)